In [13]:
from prompt_utils import build_prompt
from data_utils import create_submission, parse_predictions, create_comparison_df, read_dataset
from semevalpolar.llm import custom_rules
from semevalpolar.llm.main import test_run, create_gen, create_response
from tqdm import tqdm
from semevalpolar.utils import get_project_root

from custom_rules import apply_rules

import os
import ast

In [14]:
batch_size = 10
data_path = os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'dev', 'tel.csv')
gen = create_gen(data_path, batch_size=batch_size, randomize=True)
generator_list = list(gen)

In [15]:
df = read_dataset(data_path)

In [16]:
predictions = []
usages = []

In [17]:
for batch in tqdm(generator_list):
    response = test_run(batch, prompt_path="prompt-ds-main-classifier.txt")
    predictions.append(parse_predictions(response.output_text))
    usages.append(response.usage)

100%|██████████| 12/12 [03:46<00:00, 18.90s/it]


In [18]:
len(predictions)

12

In [19]:
flat = [x for sub in predictions for x in sub]

In [8]:
ground_truths = []

In [9]:
for i in range(len(flat) // 10):
    for x in generator_list[i]["polarization"]:
        ground_truths.append(x)

In [ ]:
texts = []

for i in range(len(predictions)):
    for j in range(batch_size):
        texts.append(generator_list[i]["text"].iloc[j])

In [ ]:
from sklearn.metrics import confusion_matrix

comparison = create_comparison_df(flat, ground_truths, texts)
cm = confusion_matrix(comparison["Ground Truth"], comparison["Predicted"], labels=[0, 1])
cm

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(comparison["Ground Truth"], comparison["Predicted"])

In [ ]:
comparison

In [ ]:
correct = comparison[comparison["Predicted"] == comparison["Ground Truth"]]
correct

In [ ]:
wrong = comparison[comparison["Ground Truth"] != comparison["Predicted"]]
wrong[wrong["Predicted"] == 1]

In [20]:
submission = create_submission(df, flat)

In [21]:
submission.to_csv(os.path.join(get_project_root(), 'predictions', 'subtask_1', 'pred_tel.csv'), index=False)